In [ ]:
import gzip
import glob
import csv
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

aa=['GLYD', 'SCP', 'SCA', 'SCV', 'SCL', 'SCI', 'SCC', 'SCM', 'SCS', 'SCT', 'SCN', 'SCQ',
    'SCF', 'SCY', 'SCW', 'SCCM', 'SCYM', 'SCE', 'SCEN', 'SCD',
    'SCDN', 'SCK', 'SCKN', 'SCR', 'SCRN', 'SCHD', 'SCHE', 'SCHP']

save_dir='../plot/'

# Load monomer/multimer rates
_rate_file = '../data/SUPP_monomer/monomer_rates_45A_9batches.dat'
_rate_df = pd.read_csv(_rate_file, sep=r'\s+', comment='#', encoding='latin-1')
monomer_rates  = dict(zip(_rate_df['SC'], _rate_df['mono_mean']))
multimer_rates = dict(zip(_rate_df['SC'], _rate_df['multi_mean']))
print(f'Loaded rates for {len(monomer_rates)} analogs from {_rate_file}')

In [ ]:
##############################################################################
# PMF mono vs total (using monomer_4.5A and total PMF data)
##############################################################################
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import os

PMF_TOTAL = '../data/pmf_data/total'
PMF_MONO = '../data/pmf_data/monomer_4.5A'
PMF_MULTI = '../data/pmf_data/multimer_4.5A'

def free_energy(pKa, charge):
    R=8.314*10**(-3) #kJ/mol.K
    T=302.3545798089109 #K
    pH=7.0
    return -charge*R*T*np.log(10)*(pKa-pH)

def get_ref_per_acid(acid):
    #source : Platzer et al. (2014)
    neutrals=['SCHP', 'SCKN', 'SCRN', 'SCDN', 'SCEN', 'SCCM', 'SCYM']
    pKas=[6.45, 10.34, 13.9, 3.86, 4.34, 8.49, 9.76]
    charges=[1, -1, -1, 1, 1, -1, -1]
    if acid in neutrals:
        zero_ref=free_energy(pKas[neutrals.index(acid)], charges[neutrals.index(acid)])
    else:
        zero_ref=0
    return zero_ref

def load_pmf_batches(acid, pmf_dir, conc=0.2):
    """
    Load PMF from 3 trajectory files (each with 3 batch columns),
    compute mean + SE over 9 batches. PMF is symmetrized (z >= 0).
    """
    batch_data = []
    for traj in [1, 2, 3]:
        if conc==0.2:
            filepath = os.path.join(pmf_dir, acid.lower(), f'trajectory{traj}.dat')
        elif conc==0.1:
            filepath = os.path.join(pmf_dir, acid.lower()+"-1", f'trajectory{traj}.dat')
        if not os.path.isfile(filepath):
            continue
        df = pd.read_csv(filepath, sep=r'\s+')
        z_col = df.columns[0]
        batch_cols = df.columns[1:]

        for col in batch_cols:
            df_batch = df[[z_col, col]].copy()
            df_batch.columns = ['z', 'pmf']
            df_batch = df_batch.dropna()
            batch_data.append(df_batch)

    if len(batch_data) == 0:
        return None

    # Merge all batches on z
    result = batch_data[0][['z']].copy()
    for i, bd in enumerate(batch_data):
        result = pd.merge(result, bd, on='z', how='outer', suffixes=('', f'_{i}'))
        if i == 0:
            result = result.rename(columns={'pmf': f'pmf_{i}'})

    pmf_cols = [c for c in result.columns if c.startswith('pmf')]
    result['PMF_mean'] = result[pmf_cols].mean(axis=1)
    result['std_error'] = result[pmf_cols].std(axis=1, ddof=1) / np.sqrt(len(pmf_cols))
    return result[['z', 'PMF_mean', 'std_error']].dropna().sort_values('z')


def plot_pmf_mono_vs_total_45A_grid(acid_list, label_list, name, titrable=False,
                                     ncols=4, x_grid_interval=10):
    """
    Grid plot comparing PMF monomer (4.5A) vs PMF total. No percentage scaling.
    """
    n_acids = len(acid_list)
    nrows = int(np.ceil(n_acids / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.2 * nrows),
                              gridspec_kw={'hspace': 0.3, 'wspace': 0.25}, sharex=True)
    axes_flat = axes.flatten()

    for idx, (acid, label) in enumerate(zip(acid_list, label_list)):
        ax = axes_flat[idx]

        data_total = load_pmf_batches(acid, PMF_TOTAL)
        data_total1 = load_pmf_batches(acid, PMF_TOTAL, conc=0.1)
        data_mono = load_pmf_batches(acid, PMF_MONO)

        if titrable:
            correction = get_ref_per_acid(acid)
        else:
            correction = 0
        if data_mono is not None:
            x_m, y_m, e_m = data_mono['z'], data_mono['PMF_mean'] + correction, data_mono['std_error']
            #ax.fill_between(x_m, y_m - e_m, y_m + e_m, alpha=0.3, color='green')
            #ax.plot(x_m, y_m, label='Contact-free', lw=1.5, color='green', linestyle='--')
            #print(e_m)

        if data_total is not None:
            x_t, y_t, e_t = data_total['z'], data_total['PMF_mean'] + correction, data_total['std_error']
            ax.fill_between(x_t, y_t - e_t, y_t + e_t, alpha=0.3, color='purple')
            ax.plot(x_t, y_t, label='0.2M', lw=1.5, color='purple')
            #print(e_t)
        
        if data_total1 is not None:
            x_t1, y_t1, e_t1 = data_total1['z'], data_total1['PMF_mean'] + correction, data_total1['std_error']
            ax.fill_between(x_t1, y_t1 - e_t1, y_t1 + e_t1, alpha=0.3, color='orange')
            ax.plot(x_t1, y_t1, label='0.1M', lw=1.5, color='orange')
            #print(e_t1)


        ax.set_xlim(0, 33)
        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.xaxis.set_major_locator(ticker.MultipleLocator(x_grid_interval))
        ax.grid(True, which='major', linestyle='--', alpha=0.5)
        ax.tick_params(axis='both', labelsize=8)

        if idx == 0:
            ax.legend(loc='upper left', fontsize=10, frameon=False, handlelength=1)

    for idx in range(n_acids, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.text(0.5, 0.03, "z (Å)", ha='center', fontsize=11)
    fig.text(0.07, 0.5, "PMF (kJ/mol)", va='center', rotation='vertical', fontsize=11)

    plt.tight_layout(rect=[0.03, 0.03, 1, 1])
    plt.savefig(save_dir + f'{name}.png', bbox_inches='tight', dpi=600)
    plt.show()


# Hydrophobic subset
aa_hydrophobic = ['SCV', 'SCL', 'SCI', 'SCM', 'SCF', 'SCY', 'SCW', 'SCRN']
labels_hydrophobic = ['VAL', 'LEU', 'ILE', 'MET', 'PHE', 'TYR$^0$', 'TRP', 'ARG$^0$']

plot_pmf_mono_vs_total_45A_grid(aa_hydrophobic, labels_hydrophobic,
                                 'FigureS11', titrable=True)

In [ ]:
##############################################################################
# Comparison of full PMF curves at 0.2 M and 0.1 M
#
# PMFs are reconstructed from the raw z-coordinate distributions. For 600-ns
# units, all frames from one independent trajectory are histogrammed before the
# PMF transformation. For 200-ns units, each trajectory is split into three
# consecutive blocks and each block distribution is transformed separately.
# Whole PMF curves are permuted over the complete reconstructed membrane-depth
# range to preserve dependence among z positions.
##############################################################################
from itertools import combinations
from scipy import stats

ALPHA = 0.05
PERMUTATION_BLOCK_NS = 600  #for tableS4
#PERMUTATION_BLOCK_NS = 200  #for tableS5
PERMUTATION_STATISTIC = 'RMS'

RAW_DATA_DIR = '../data/distribution_data/raw_data'
TRAJECTORIES = (1, 2, 3)
PRODUCTION_NS = 600
FRAMES_PER_NS = 100

K_B = 0.008314
T_PMF = 303.15
EPSILON = 1e-5
Z_BIN = 1.0
Z_MAX = 50.0
REF_LO, REF_HI = 40.0, 50.0
BIN_EDGES = np.arange(0.0, Z_MAX + Z_BIN, Z_BIN)
BIN_CENTERS = 0.5 * (BIN_EDGES[:-1] + BIN_EDGES[1:])
REF_MASK = (BIN_CENTERS >= REF_LO) & (BIN_CENTERS <= REF_HI)

if PERMUTATION_BLOCK_NS not in (200, 600):
    raise ValueError('PERMUTATION_BLOCK_NS must be 200 or 600.')
if PRODUCTION_NS % PERMUTATION_BLOCK_NS != 0:
    raise ValueError('PERMUTATION_BLOCK_NS must divide PRODUCTION_NS exactly.')

def load_raw_z_trajectory(acid, trajectory, conc):
    analog = acid.lower() if conc == 0.2 else acid.lower() + '-1'
    filepath = os.path.join(
        RAW_DATA_DIR, analog, f'{analog}_contacts_{trajectory}.dat'
    )
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f'Raw trajectory not found: {filepath}')

    frame_z = np.loadtxt(filepath, comments='#', usecols=(0, 4))
    frames = frame_z[:, 0].astype(np.int64)
    z_values = frame_z[:, 1]
    expected_last_frame = PRODUCTION_NS * FRAMES_PER_NS - 1
    if frames.min() != 0 or frames.max() != expected_last_frame:
        raise ValueError(
            f'{filepath} spans frames {frames.min()}-{frames.max()}; '
            f'expected 0-{expected_last_frame}.'
        )
    return frames, z_values

def pmf_from_z_distribution(z_values):
    counts, _ = np.histogram(np.abs(z_values), bins=BIN_EDGES)
    if counts.sum() == 0:
        raise ValueError('Cannot calculate a PMF from an empty distribution.')

    probability = counts / counts.sum()
    pmf = -K_B * T_PMF * np.log(probability + EPSILON)
    pmf -= pmf[REF_MASK].mean()
    return pmf

def trajectory_block_pmfs(frames, z_values):
    block_frames = PERMUTATION_BLOCK_NS * FRAMES_PER_NS
    n_blocks = PRODUCTION_NS // PERMUTATION_BLOCK_NS
    pmfs = []

    for block_index in range(n_blocks):
        first_frame = block_index * block_frames
        last_frame = (block_index + 1) * block_frames
        in_block = (frames >= first_frame) & (frames < last_frame)
        if not in_block.any():
            raise ValueError(
                f'No raw observations in frame interval '
                f'[{first_frame}, {last_frame}).'
            )
        pmfs.append(pmf_from_z_distribution(z_values[in_block]))

    return pmfs

def load_raw_pmf_blocks(acid, conc):
    pmf_curves = []
    for trajectory in TRAJECTORIES:
        frames, z_values = load_raw_z_trajectory(acid, trajectory, conc)
        pmf_curves.extend(trajectory_block_pmfs(frames, z_values))

    expected_curves = len(TRAJECTORIES) * (PRODUCTION_NS // PERMUTATION_BLOCK_NS)
    if len(pmf_curves) != expected_curves:
        raise ValueError(
            f'Constructed {len(pmf_curves)} PMFs; expected {expected_curves}.'
        )
    return BIN_CENTERS.copy(), np.column_stack(pmf_curves)

def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted_ranked = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted

def se_weighted_statistic(group_a, group_b):
    mean_diff = group_a.mean(axis=1) - group_b.mean(axis=1)
    se_a = group_a.std(axis=1, ddof=1) / np.sqrt(group_a.shape[1])
    se_b = group_b.std(axis=1, ddof=1) / np.sqrt(group_b.shape[1])
    se_diff = np.sqrt(se_a**2 + se_b**2)
    valid = np.isfinite(se_diff) & (se_diff > 0)
    return float(np.mean((mean_diff[valid] / se_diff[valid]) ** 2))

def rms_statistic(group_a, group_b):
    mean_diff = group_a.mean(axis=1) - group_b.mean(axis=1)
    return float(np.sqrt(np.mean(mean_diff**2)))

STATISTIC_FUNCTIONS = {
    'SE': se_weighted_statistic,
    'RMS': rms_statistic,
}
if PERMUTATION_STATISTIC not in STATISTIC_FUNCTIONS:
    raise ValueError(
        f'PERMUTATION_STATISTIC must be one of {tuple(STATISTIC_FUNCTIONS)}.'
    )

def exact_curve_permutation_test(group_a, group_b, statistic_function):
    pooled = np.concatenate([group_a, group_b], axis=1)
    n_a = group_a.shape[1]
    all_indices = np.arange(pooled.shape[1])
    observed = statistic_function(group_a, group_b)
    permutation_stats = []

    for selected in combinations(all_indices, n_a):
        selected = np.asarray(selected)
        other = np.setdiff1d(all_indices, selected, assume_unique=True)
        permutation_stats.append(
            statistic_function(pooled[:, selected], pooled[:, other])
        )

    permutation_stats = np.asarray(permutation_stats)
    p_value = np.mean(permutation_stats >= observed - 1e-12)
    return observed, float(p_value), int(permutation_stats.size)

def compare_curves(acid):
    z_02, blocks_02 = load_raw_pmf_blocks(acid, conc=0.2)
    z_01, blocks_01 = load_raw_pmf_blocks(acid, conc=0.1)
    z_common = np.intersect1d(z_02, z_01)
    if z_common.size < 3:
        return None

    group_02 = blocks_02[np.searchsorted(z_02, z_common), :]
    group_01 = blocks_01[np.searchsorted(z_01, z_common), :]
    mean_02 = group_02.mean(axis=1)
    mean_01 = group_01.mean(axis=1)
    mean_diff = mean_02 - mean_01

    _, p_mean = stats.ttest_rel(mean_02, mean_01)
    statistic_function = STATISTIC_FUNCTIONS[PERMUTATION_STATISTIC]
    permutation_stat, p_perm, n_permutations = exact_curve_permutation_test(
        group_02, group_01, statistic_function
    )

    return {
        'acid': acid,
        'source': 'raw z distributions',
        'block_ns': PERMUTATION_BLOCK_NS,
        'n_blocks_per_concentration': group_02.shape[1],
        'K': z_common.size,
        'z_min (A)': float(z_common.min()),
        'z_max (A)': float(z_common.max()),
        'mean_diff (kJ/mol)': float(mean_diff.mean()),
        'p_mean_raw': float(p_mean),
        f'T_{PERMUTATION_STATISTIC}': permutation_stat,
        f'p_perm_{PERMUTATION_STATISTIC}_raw': p_perm,
        'n_perm': n_permutations,
    }

results = []
for acid, label in zip(aa_hydrophobic, labels_hydrophobic):
    result = compare_curves(acid)
    if result is not None:
        result['label'] = label
        results.append(result)

statistic_column = f'T_{PERMUTATION_STATISTIC}'
raw_p_column = f'p_perm_{PERMUTATION_STATISTIC}_raw'
fdr_p_column = f'p_perm_{PERMUTATION_STATISTIC}_FDR'
test_column = f'{PERMUTATION_STATISTIC} test'

df_stats = pd.DataFrame(results).set_index('label')
df_stats['p_mean_FDR'] = bh_adjust(df_stats['p_mean_raw'])
df_stats['mean test'] = np.where(
    df_stats['p_mean_FDR'] < ALPHA, 'different', 'not different'
)
df_stats[fdr_p_column] = bh_adjust(df_stats[raw_p_column])
df_stats[test_column] = np.where(
    df_stats[fdr_p_column] < ALPHA, 'different', 'not different'
)

mean_table = df_stats[[
    'acid', 'mean_diff (kJ/mol)', 'p_mean_raw', 'p_mean_FDR', 'mean test'
]]
permutation_table = df_stats[[
    'acid', 'source', 'block_ns', 'n_blocks_per_concentration',
    'K', 'z_min (A)', 'z_max (A)', statistic_column, raw_p_column,
    fdr_p_column, test_column, 'n_perm'
]]

with pd.option_context('display.float_format', '{:.3g}'.format):
    print('Test 1 - paired t-test on mean curves (exploratory):')
    print(mean_table.to_string())
    print(
        f'\nTest 2 - exact whole-curve permutation test using '
        f'T_{PERMUTATION_STATISTIC} ({PERMUTATION_BLOCK_NS}-ns raw distributions):'
    )
    print(permutation_table.to_string())

mean_different = df_stats.index[df_stats['p_mean_FDR'] < ALPHA].tolist()
permutation_different = df_stats.index[df_stats[fdr_p_column] < ALPHA].tolist()

print(
    f"\nComplete tested range: {df_stats['z_min (A)'].min():g}-"
    f"{df_stats['z_max (A)'].max():g} A over {int(df_stats['K'].min())} bins."
)
print(
    f"Test t apparié des courbes moyennes: après correction FDR, "
    f"{len(mean_different)}/{len(df_stats)} analogues diffèrent "
    f"({', '.join(mean_different) if mean_different else 'aucun'})."
)
print(
    f"Test global par permutation avec T_{PERMUTATION_STATISTIC} "
    f"({PERMUTATION_BLOCK_NS}-ns raw distributions): après correction FDR, "
    f"{len(permutation_different)}/{len(df_stats)} analogues diffèrent "
    f"({', '.join(permutation_different) if permutation_different else 'aucun'})."
)
print(
    "Conclusion: chaque PMF est calculé après agrégation des positions z dans "
    "l'unité temporelle choisie; les courbes PMF entières sont ensuite permutées."
)

output_file = ( f'TableS4.csv')
#output_file = ( f'TableS5.csv')

df_stats.to_csv(os.path.join(save_dir, output_file))